# 02 · Análisis exploratorio

**Clase 4:** distribución, boxplot, dispersión y matriz de correlación. **Clase 3:** cobertura y ceros antes de modelar. Todos los gráficos usan meses-parque observados.

In [ ]:
from pathlib import Path
import os, sys
if 'google.colab' in sys.modules:
    import subprocess
    root = Path('/content/tpo-solar')
    if not root.exists():
        subprocess.run(['git','clone','https://github.com/imelinc/tpo-solar.git',str(root)],check=True)
    os.chdir(root)
    subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements.txt'],check=True)
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.config import DATA_RAW, DATA_PROCESSED
import json
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

In [ ]:
B = pd.read_parquet(DATA_PROCESSED / 'dataset_B_argentina_mensual.parquet')
from src.academico import preparar
d = preparar(B)
print('Meses-parque comparables:', len(d), '| parques:', d['central'].nunique(), '| provincias:', d['provincia'].nunique())

## Distribución del factor de capacidad

El factor es la energía del mes dividida por la potencia y por las horas del mes. Permite comparar parques de distinto tamaño.

In [ ]:
px.histogram(d, x='factor_capacidad', nbins=30, title='Clase 4 · Distribución del factor de capacidad mensual',
             labels={'factor_capacidad':'Factor de capacidad'}).show()
print(d['factor_capacidad'].describe().round(4))

## Estacionalidad

El boxplot por mes muestra mediana, cuartiles y extremos. Un valor extremo es una observación para investigar: no confirma una falla.

In [ ]:
px.box(d, x='mes', y='factor_capacidad', points='outliers', title='Clase 4 · Factor de capacidad por mes',
       labels={'mes':'Mes','factor_capacidad':'Factor de capacidad'}).show()

## Sol contra rendimiento

La dispersión y la correlación describen asociaciones. No confirman causalidad ni anticipan el error de un modelo en datos nuevos.

In [ ]:
px.scatter(d, x='om_ghi_kwh_m2_mes', y='factor_capacidad', color='provincia', opacity=.45,
           labels={'om_ghi_kwh_m2_mes':'Irradiación del mes (kWh/m²)','factor_capacidad':'Factor de capacidad'},
           title='Clase 4 · Irradiación contra factor de capacidad').show()
cols = ['factor_capacidad','om_ghi_kwh_m2_mes','om_temp_max_media_c','om_nubosidad_media_pct','lat','lon']
corr = d[cols].corr().round(2)
px.imshow(corr, text_auto=True, zmin=-1, zmax=1, color_continuous_scale='RdBu_r',
          title='Clase 4 · Matriz de correlación (Pearson)').show()

## Cobertura y ceros

Las ubicaciones inferidas de una localidad son aproximadas. Un mes con cero posterior al inicio operativo se conserva; un mes sin reporte no aparece en la tabla y no se convierte en cero.

In [ ]:
print(d.groupby('provincia').agg(parques=('central','nunique'), meses=('fecha','size'), factor_medio=('factor_capacidad','mean')).sort_values('parques', ascending=False).round(3))
print('Ceros por parque:')
print(d[d['energia_mwh'].eq(0)].groupby('central')['fecha'].apply(list))

## Hallazgos para el modelado

- La irradiación del mes es la variable más asociada al factor, pero la nube es ancha: hacen falta ubicación, temperatura, nubosidad y mes.
- El factor baja en invierno; el mes entra al modelo en one-hot.
- Los parques de Chaco, Formosa y Neuquén entraron después de 2023 y quedan fuera del rango de entrenamiento.